In [5]:
import logging
import boto3
import json
import pandas as pd
from botocore.exceptions import ClientError

Crea un notebook flights_analytics.ipynb en tu repositorio. El notebook debe:

Conectarse a la Read Replica con SQLAlchemy y pd.read_sql

Ejecutar las 8 preguntas (P1–P5 y W1–W3) como queries SQL

Mostrar el resultado de cada query como DataFrame de pandas

Incluir al menos una visualización por pregunta usando cualquiera de las siguientes librerías:

matplotlib — gráficas estáticas (plt.style.use('ggplot'))
plotnine — gramática de gráficas estilo ggplot2
plotly — gráficas interactivas
great_tables — tablas formateadas con resaltado, colores y barras inline
Elige la herramienta que mejor comunique la respuesta de cada pregunta: barras para rankings (P1, P2, P5), tablas formateadas para conteos (P3), líneas para series de tiempo (P4, W2), y tablas con ranking para W1 y W3.

In [6]:
# ──────────────────────────────────────────────
# Configuración del logger
# ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

In [7]:
def get_secret():

    secret_name = "itam/rds/flights/credentials"
    region_name = "us-east-1"

    # Create a Secrets Manager client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )

    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        logger.error(f"Error retrieving secret: {e}")
        raise e

    secret = get_secret_value_response['SecretString']
    return json.loads(secret)

creds   = get_secret()

In [ ]:
#conectarse a la base de datos con SQLAlchemy
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

# RDS endpoint Replica
RDS_ENDPOINT = "itam-flights-replica-956463123241.cmvsysimg4pq.us-east-1.rds.amazonaws.com"
DB_NAME      = creds["dbname"]
DB_USER      = creds["username"]
DB_PASSWORD  = creds["password"]
DB_PORT      = creds["port"]

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{RDS_ENDPOINT}:{DB_PORT}/{DB_NAME}"
)
logger.info("Conexión a la base de datos establecida con SQLAlchemy")

# Verificar conexión
try:
    with engine.connect() as conn:
        print("✓ Conexión exitosa a RDS PostgreSQL")
except SQLAlchemyError as e:
    print(f"✗ Error de conexión: {e}")

2026-04-12 22:14:32,382 - INFO - Conexión a la base de datos establecida con SQLAlchemy


✓ Conexión exitosa a RDS PostgreSQL


# 6.2 Preguntas con SELECT
Ejecuta cada query en DBeaver y toma un screenshot del resultado mostrando tanto la query como los resultados en la misma pantalla.

In [9]:
# ──────────────────────────────────────────────
# Consultas de prueba
# ──────────────────────────────────────────────

# P1. ¿Cuáles son las 10 rutas (origen → destino) con mayor número de vuelos? 
query_P1 = """
SELECT origin_airport, destination_airport,
	   COUNT(*) AS numero_vuelos_por_ruta
FROM flights
GROUP BY origin_airport, destination_airport
ORDER BY numero_vuelos_por_ruta DESC
LIMIT 10;
"""

#Mostrar el resultado de cada query como DataFrame de pandas

with engine.connect() as conn:
    result_P1 = pd.read_sql(query_P1, conn)
    logger.info("✓ Consulta P1 ejecutada con éxito")
    display(result_P1)

2026-04-12 22:14:34,850 - INFO - ✓ Consulta P1 ejecutada con éxito


,origin_airport,destination_airport,numero_vuelos_por_ruta
0,LAX,JFK,1218
1,JFK,LAX,1217
2,SFO,LAX,1186
3,LAX,SFO,1172
4,LAS,LAX,1018
5,LAX,LAS,987
6,LGA,ORD,917
7,ORD,LGA,908
8,HNL,OGG,874
9,OGG,HNL,873


In [ ]:
!pip install great_tables -q

from great_tables import GT, loc, style

In [ ]:
# Visualización P1: Barras con Plotly (rankings)
import plotly.express as px

result_P1['ruta'] = result_P1['origin_airport'] + ' → ' + result_P1['destination_airport']

fig_p1 = px.bar(
    result_P1,
    x='numero_vuelos_por_ruta',
    y='ruta',
    orientation='h',
    title='Top 10 Rutas con Mayor Número de Vuelos',
    labels={'numero_vuelos_por_ruta': 'Número de Vuelos', 'ruta': 'Ruta'},
    color='numero_vuelos_por_ruta',
    color_continuous_scale='Reds'
)
fig_p1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_p1.show()

In [11]:
# P2. ¿Cuáles son las 5 aerolíneas con mayor porcentaje de vuelos cancelados?
# Incluye el código de aerolínea, el total de vuelos, el total de cancelaciones y el porcentaje.
query_p2 = """
SELECT
    airline,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) AS cancelled_flights,
    SUM(CASE WHEN cancelled = 1 THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS avg_cancellations
FROM flights
GROUP BY airline
order by avg_cancellations desc
limit 5;
"""

with engine.connect() as conn:
    result_P2 = pd.read_sql(query_p2, conn)
    logger.info("✓ Consulta P2 ejecutada con éxito")
    display(result_P2)


2026-04-12 22:16:13,329 - INFO - ✓ Consulta P2 ejecutada con éxito


,airline,total_flights,cancelled_flights,avg_cancellations
0,MQ,31896,3136,0.098320
1,B6,23062,1479,0.064131
2,EV,52965,2523,0.047635
3,US,35591,1268,0.035627
4,UA,40873,1424,0.034840


In [ ]:
# Visualización P2: Barras con Plotly (rankings)
result_P2['pct_cancellations'] = result_P2['avg_cancellations'] * 100

fig_p2 = px.bar(
    result_P2,
    x='airline',
    y='pct_cancellations',
    title='Top 5 Aerolíneas con Mayor Porcentaje de Cancelaciones',
    labels={'pct_cancellations': '% Cancelaciones', 'airline': 'Aerolínea'},
    color='pct_cancellations',
    color_continuous_scale='Blues',
    text='pct_cancellations'
)
fig_p2.update_traces(texttemplate='%{text:.2f}%', textposition='outside')
fig_p2.update_layout(yaxis_title='Porcentaje de Cancelaciones (%)')
fig_p2.show()

In [12]:
#P3. ¿Cuántos vuelos fueron cancelados por cada causa (CANCELLATION_REASON)? 
# Muestra la causa y el conteo. Los vuelos no cancelados tienen CANCELLATION_REASON = NULL — exclúyelos filtrando WHERE cancellation_reason IS NOT NULL.

query_p3 = """
select 
	cancellation_reason,
	COUNT(*) as count_cancellation_number
from FLIGHTS
where cancellation_reason is not null
group by cancellation_reason
order by count_cancellation_number desc;"""

with engine.connect() as conn:
    result_P3 = pd.read_sql(query_p3, conn)
    logger.info("✓ Consulta P3 ejecutada con éxito")
    display(result_P3)

2026-04-12 22:17:01,317 - INFO - ✓ Consulta P3 ejecutada con éxito


,cancellation_reason,count_cancellation_number
0,B,10630
1,A,3230
2,C,2962
3,D,2


In [ ]:
(
    GT(result_P3)
    .tab_header(
        title="Vuelos Cancelados por Causa",
        subtitle="Distribución de razones de cancelación"
    )
    .fmt_number(columns="count_cancellation_number", use_seps=True)
    .data_color(
        columns="count_cancellation_number",
        palette=["#f7fbff", "#08306b"]  # azul claro a oscuro
    )
)

In [13]:
"""P4. ¿Cuál es el retraso promedio de salida por mes, considerando únicamente los vuelos que efectivamente se retrasaron?
Incluye solo vuelos no cancelados y con retraso de salida positivo: WHERE cancelled = 0 AND departure_delay > 0. Ordena por mes."""

query_p4 = """
select
	sum(departure_delay) as sum_departure_delay ,
	AVG(departure_delay) as avg_departure_delay,
	COUNT(*) as count_delay_per_month
from flights
WHERE cancelled = 0 AND departure_delay > 0
group by month;
"""

with engine.connect() as conn:
    result_P4 = pd.read_sql(query_p4, conn)
    logger.info("✓ Consulta P4 ejecutada con éxito")
    display(result_P4)

2026-04-12 22:17:37,447 - INFO - ✓ Consulta P4 ejecutada con éxito


,sum_departure_delay,avg_departure_delay,count_delay_per_month
0,5746183.0,32.571225,176419
1,475398.0,42.306487,11237


In [ ]:
# Visualización P4: Línea con Matplotlib (serie de tiempo)
import matplotlib.pyplot as plt

plt.style.use('ggplot')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(result_P4.index + 1, result_P4['avg_departure_delay'], marker='o', linewidth=2, color='#CB181D')
ax.fill_between(result_P4.index + 1, result_P4['avg_departure_delay'], alpha=0.3, color='#CB181D')
ax.set_xlabel('Mes')
ax.set_ylabel('Retraso Promedio (minutos)')
ax.set_title('Retraso Promedio de Salida por Mes\n(Solo vuelos no cancelados con retraso positivo)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'])
plt.tight_layout()
plt.show()

In [14]:
# P5. ¿Cuáles son los 10 aeropuertos de origen con más minutos totales de retraso atribuidos al clima (WEATHER_DELAY)? 
# Muestra el código del aeropuerto y la suma total de minutos.
query_p5 = """
select
	origin_airport,
	sum(weather_delay) as sum_weather_delay
from flights 
where weather_delay is not null
group by origin_airport
order by sum_weather_delay desc
limit 10;"""

with engine.connect() as conn:
    result_P5 = pd.read_sql(query_p5, conn)
    logger.info("✓ Consulta P5 ejecutada con éxito")
    display(result_P5)

2026-04-12 22:18:11,391 - INFO - ✓ Consulta P5 ejecutada con éxito


,origin_airport,sum_weather_delay
0,ORD,102749.0
1,JFK,20542.0
2,LGA,13620.0
3,ATL,12562.0
4,DTW,12361.0
5,BOS,11361.0
6,DEN,8359.0
7,DCA,8188.0
8,PHX,7878.0
9,DFW,6870.0


In [ ]:
# Visualización P5: Barras con Plotly (rankings)
fig_p5 = px.bar(
    result_P5,
    x='sum_weather_delay',
    y='origin_airport',
    orientation='h',
    title='Top 10 Aeropuertos con Mayor Retraso por Clima',
    labels={'sum_weather_delay': 'Total Minutos de Retraso', 'origin_airport': 'Aeropuerto'},
    color='sum_weather_delay',
    color_continuous_scale='Greens'
)
fig_p5.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_p5.show()

# Preguntas con Window Functions
Ejecutar las 8 preguntas (P1–P5 y W1–W3) como queries SQL

In [ ]:
# W1. Para cada aerolínea, ¿cuál fue el vuelo con el mayor retraso de llegada?
# Usa RANK() particionando por aerolínea, ordenando por ARRIVAL_DELAY DESC NULLS LAST. 
# Devuelve solo el rank 1 de cada aerolínea.

In [17]:
query_w1 = """
select *
from (
select
	airline,
	flight_number,
	arrival_delay,
	origin_airport,
	destination_airport,
	RANK() over(
	partition by airline
	order by arrival_delay  desc nulls LAST
	) as most_delay_flight_per_airline
from flights
)
where most_delay_flight_per_airline =1
order by arrival_delay desc;
"""

with engine.connect() as conn:
    result_W1 = pd.read_sql(query_w1, conn)
    logger.info("✓ Consulta W1 ejecutada con éxito")
    display(result_W1)

2026-04-12 22:57:03,900 - INFO - ✓ Consulta W1 ejecutada con éxito


,airline,flight_number,arrival_delay,origin_airport,destination_airport,most_delay_flight_per_airline
0,AA,1322,1971.0,BHM,DFW,1
1,DL,1435,1174.0,MIA,MSP,1
2,HA,51,1013.0,JFK,HNL,1
3,OO,2642,953.0,CHO,ORD,1
4,UA,1443,863.0,ICT,DEN,1
5,MQ,2962,788.0,CLL,DFW,1
6,EV,2567,723.0,SHV,DFW,1
7,F9,1256,686.0,ORD,MIA,1
8,US,1782,621.0,SEA,CLT,1
9,WN,259,593.0,IAD,DEN,1


In [ ]:
# great table result_W1
(
    GT(result_W1)
    .tab_header(
        title="Vuelo con Mayor Retraso de Llegada por Aerolínea",
        subtitle="Ranking de vuelos más retrasados por aerolínea"
    )
    .cols_label(
        airline="Aerolínea",
        flight_number="Número de Vuelo",
        arrival_delay="Retraso de Llegada (min)",
        origin_airport="Aeropuerto Origen",
        destination_airport="Aeropuerto Destino"
    )
    .fmt_number(columns="arrival_delay", use_seps=True)
    .data_color(
        columns="arrival_delay",
        palette=["#fff5f0", "#cb181d"]  # rojo claro a oscuro
    )
)

In [28]:
import awswrangler as wr
DATABASE_NAME = "flights_silver"
TABLE_NAME = "flights_monthly"
query_test = "SELECT * FROM flights_monthly LIMIT 5"

#SELECT * FROM "flights_silver"."flights_monthly" limit 10;

df_test = wr.athena.read_sql_query(
    query_test,
    database=DATABASE_NAME,
    ctas_approach=False,
)

df_test

,month,airline,total_flights,total_delayed,total_cancelled,avg_arrival_delay,on_time_pct
0,1,AA,44059,16558,900,6.955843,78.731950
1,1,AS,13257,3595,64,-0.320888,85.894609
2,1,B6,21623,7858,1102,7.347281,76.711259
3,1,DL,64421,19477,678,-2.043847,87.770903
4,1,EV,49925,16090,1702,8.537497,77.888695


In [27]:
# W2
# W2. ¿Cuál fue la variación mes a mes en el total de vuelos durante 2015? 
# Usa LAG() para calcular la diferencia absoluta y el cambio porcentual respecto al mes anterior. 
# Ordena por mes.

query_w2 = """
WITH vuelos_por_mes AS (
    SELECT
        month,
        SUM(total_flights) AS total_vuelos
    FROM flights_monthly
    GROUP BY month
)
SELECT
    month,
    total_vuelos,
    LAG(total_vuelos) OVER (ORDER BY month) AS vuelos_mes_anterior,
    total_vuelos - LAG(total_vuelos) OVER (ORDER BY month) AS diferencia_absoluta,
    ROUND(
        (total_vuelos - LAG(total_vuelos) OVER (ORDER BY month)) * 100.0
        / LAG(total_vuelos) OVER (ORDER BY month), 2
    ) AS cambio_porcentual
FROM vuelos_por_mes
ORDER BY month;
"""

df_w2 = wr.athena.read_sql_query(
    query_w2,
    database=DATABASE_NAME,
    ctas_approach=False,
)

df_w2

,month,total_vuelos,vuelos_mes_anterior,diferencia_absoluta,cambio_porcentual
0,1,469968,<NA>,<NA>,NaN
1,2,429191,469968,-40777,-8.68
2,3,504312,429191,75121,17.50
3,4,485151,504312,-19161,-3.80
4,5,496993,485151,11842,2.44
5,6,503897,496993,6904,1.39
6,7,520718,503897,16821,3.34
7,8,510536,520718,-10182,-1.96
8,9,464946,510536,-45590,-8.93
9,10,486165,464946,21219,4.56


In [ ]:
# Visualización W2: Línea con Plotly (serie de tiempo)
fig_w2 = px.line(
    df_w2,
    x='month',
    y='total_vuelos',
    title='Variación Mes a Mes en el Total de Vuelos (2015)',
    labels={'month': 'Mes', 'total_vuelos': 'Total de Vuelos'},
    markers=True
)

# Agregar línea de cambio porcentual en eje secundario
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_w2_dual = make_subplots(specs=[[{"secondary_y": True}]])

fig_w2_dual.add_trace(
    go.Scatter(x=df_w2['month'], y=df_w2['total_vuelos'], name='Total Vuelos', 
               mode='lines+markers', line=dict(color='#31A354', width=2)),
    secondary_y=False
)

fig_w2_dual.add_trace(
    go.Bar(x=df_w2['month'], y=df_w2['cambio_porcentual'], name='% Cambio',
           marker_color='#74C476', opacity=0.6),
    secondary_y=True
)

fig_w2_dual.update_layout(
    title='Variación Mes a Mes en el Total de Vuelos',
    xaxis_title='Mes'
)
fig_w2_dual.update_yaxes(title_text="Total Vuelos", secondary_y=False)
fig_w2_dual.update_yaxes(title_text="Cambio Porcentual (%)", secondary_y=True)

fig_w2_dual.show()

In [20]:
# W3. Para el aeropuerto LAX el día 2015-01-01, ¿cuáles fueron los primeros 5 vuelos según el horario de salida programado? 
# Usa ROW_NUMBER() particionando por (year, month, day, origin_airport) y ordenando por SCHEDULED_DEPARTURE ASC. 
# Muestra el número de vuelo, aerolínea, destino y horario.
query_w3 = """
with vuelos_lax as (
select 
	flight_number,
	airline,
	year,
	month,
	day,
	origin_airport,
	destination_airport,
	scheduled_departure,
	ROW_NUMBER() over(
	partition by year,month,day,origin_airport
	order by scheduled_departure asc
	) AS rn
from flights 
where origin_airport = 'LAX' and year = '2015' and "month" = 1 and "day" = 1
)
select 
	rn,
	flight_number,
    airline,
    destination_airport,
    scheduled_departure
from vuelos_lax 
where rn <=5
order by rn"""
with engine.connect() as conn:
    result_W3 = pd.read_sql(query_w3, conn)
    logger.info("✓ Consulta W3 ejecutada con éxito")
    display(result_W3)

2026-04-12 23:09:16,796 - INFO - ✓ Consulta W1 ejecutada con éxito


,rn,flight_number,airline,destination_airport,scheduled_departure
0,1,2336,AA,PBI,10
1,2,258,AA,MIA,20
2,3,2013,US,CLT,30
3,4,1434,DL,MSP,35
4,5,115,AA,MIA,105


In [ ]:
# great table result_W3
(
    GT(result_W3)
    .tab_header(
        title="Primeros 5 Vuelos desde LAX el 2015-01-01",
        subtitle="Ordenados por horario de salida programado"
    )
    .cols_label(
        rn="Ranking",
        flight_number="Número de Vuelo",
        airline="Aerolínea",
        destination_airport="Destino",
        scheduled_departure="Salida Programada"
    )
    .fmt_number(columns="scheduled_departure", use_seps=True)
    .data_color(
        columns="rn",
        palette=["#f7fbff", "#08306b"]  # azul claro a oscuro
    )
)